# Exercise 09: Adversarial Training

**Goal:** Implement adversarial training to produce a model robust against FGSM, and observe the clean/adversarial accuracy tradeoff.

**Time:** ~10 min

**Framework:** TensorFlow/Keras (contrast with PyTorch used in earlier exercises).

## Background

**Adversarial training** augments the standard training loop by mixing clean and adversarial examples in each batch:

$$\mathcal{L}_\text{total} = \frac{\mathcal{L}_\text{clean} + \mathcal{L}_\text{adv}}{2}$$

The key challenge: gradients must flow through the *attack step* (to generate adversarial examples) AND through the *model parameters* (to update the model). TensorFlow's `tf.GradientTape` makes this explicit.

**Robustness–accuracy tradeoff:** robust models sacrifice ~2–3% clean accuracy to gain ~70% adversarial accuracy on MNIST.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import helper

tf.random.set_seed(42)
np.random.seed(42)

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test  = x_test.astype(np.float32)  / 255.0
x_train = np.expand_dims(x_train, -1)
x_test  = np.expand_dims(x_test,  -1)
y_train_cat = tf.keras.utils.to_categorical(y_train, 10)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  10)

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_cat)).shuffle(10000).batch(64)
test_ds  = tf.data.Dataset.from_tensor_slices((x_test,  y_test_cat)).batch(256)

def create_model():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10,  activation='softmax'),
    ])

# Train standard model
print("Training standard model (5 epochs)...")
model = create_model()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, epochs=5, validation_data=test_ds, verbose=1)

# Helper: gradient of loss w.r.t. input (pre-written)
def compute_gradient_network(image, label, model):
    image = tf.convert_to_tensor(image, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image)
        prediction = model(image)
        loss = tf.keras.losses.categorical_crossentropy(label, prediction)
    return tape.gradient(loss, image)

## Warm-up: GradientTape for Input Gradients

`tf.GradientTape` records operations for automatic differentiation. By calling `tape.watch(x)` on a non-variable tensor, we can compute gradients w.r.t. inputs — the mechanism behind FGSM.

In [ ]:
# 🎯 WARM-UP TODO: Use tf.GradientTape to compute d/dx[cos(x^2)] at x = 42.
# d/dx cos(x^2) = -2x * sin(x^2)
# At x=42: -2*42*sin(1764) ≈ -84 * sin(1764)
# sin(1764 rad) ≈ -1  (1764 ≈ 3π/2 + 280*2π), so expected result ≈ +84.0

x = tf.constant(42.0)
with tf.GradientTape() as tape:
    # TODO: watch x and compute cos(x * x)
    # tape.watch(x)
    # y = ...
    pass

dy_dx = None  # TODO: tape.gradient(y, x)
print(f"d/dx cos(x^2) at x=42: {dy_dx}")
print("Expected: approximately +84.0")

## 🎯 TODO: Implement FGSM (TensorFlow)

In [ ]:
def fgsm_attack(image, label, model, epsilon=0.2):
    """
    FGSM: $x_\\text{adv} = \\text{clip}(x + \\varepsilon \\cdot \\text{sign}(\\nabla_x \\mathcal{L}), 0, 1)$

    Steps:
    # TODO: 1. gradient = compute_gradient_network(image, label, model)
    # TODO: 2. perturbation = epsilon * tf.sign(gradient)
    # TODO: 3. adversarial_image = image + perturbation
    # TODO: 4. adversarial_image = tf.clip_by_value(adversarial_image, 0.0, 1.0)
    # TODO: 5. Return adversarial_image
    """
    adversarial_image = None  
    return adversarial_image

def evaluate_adversarial(model, dataset, epsilon=0.2):
    total = correct = 0
    for images, labels in dataset:
        adv = fgsm_attack(images, labels, model, epsilon)
        preds = model(adv, training=False)
        correct += int(np.sum(np.argmax(preds.numpy(), 1) == np.argmax(labels.numpy(), 1)))
        total   += labels.shape[0]
    print(f"Adversarial accuracy (ε={epsilon}): {100*correct/total:.2f}%")

print("Standard model on clean data:")
model.evaluate(test_ds, verbose=1)
print("Standard model on adversarial data:")
evaluate_adversarial(model, test_ds, epsilon=0.2)

## Standard Training Loop (pre-written)

Study this before implementing adversarial training — the pattern is the same, extended with adversarial examples.

In [ ]:
# Pre-written standard training loop (study this before implementing adversarial training)
def standard_training(model, train_ds, epochs=5):
    optimizer = tf.keras.optimizers.Adam()
    loss_fn   = tf.keras.losses.CategoricalCrossentropy()
    for epoch in range(epochs):
        print(f"  Epoch {epoch+1}/{epochs}")
        for images, labels in train_ds:
            with tf.GradientTape() as tape:
                predictions = model(images, training=True)
                loss = loss_fn(labels, predictions)
            # Gradients w.r.t. model WEIGHTS (not inputs)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))


## 🎯 TODO: Implement Adversarial Training

**Hint:** The training step is identical to `standard_training` except you compute $\mathcal{L}_\text{total} = (\mathcal{L}_\text{clean} + \mathcal{L}_\text{adv}) / 2$ and backpropagate through it.

In [ ]:
def adversarial_training(model, train_ds, epsilon=0.2, epochs=5):
    """
    Extend standard_training to perform adversarial training.

    For each batch:
    # 1. adv_images = fgsm_attack(images, labels, model, epsilon)
    # 2. Open a GradientTape (watches model.trainable_variables automatically)
    # 3. loss_clean = loss_fn(labels, model(images, training=True))
    #             loss_adv   = loss_fn(labels, model(adv_images, training=True))
    # 4. total_loss = (loss_clean + loss_adv) / 2
    # 5. Backpropagate and apply optimizer (same as standard_training)

    IMPORTANT: fgsm_attack uses GradientTape internally, but with tape.watch(image) —
    it does NOT interfere with the outer GradientTape that records model weights.
    """
    optimizer = tf.keras.optimizers.Adam()
    loss_fn   = tf.keras.losses.CategoricalCrossentropy()
    for epoch in range(epochs):
        print(f"  Epoch {epoch+1}/{epochs}")
        for images, labels in train_ds:
            # 🎯 TODO: implement adversarial training step
            pass
        evaluate_adversarial(model, test_ds, epsilon)

print("Training robust model with adversarial training (5 epochs)...")
robust_model = create_model()
adversarial_training(robust_model, train_ds, epsilon=0.2, epochs=5)

## Compare Standard vs Robust Model

In [ ]:
# Compare clean and adversarial accuracy for both models
def clean_accuracy(model, dataset):
    total = correct = 0
    for images, labels in dataset:
        preds = model(images, training=False)
        correct += int(np.sum(np.argmax(preds.numpy(), 1) == np.argmax(labels.numpy(), 1)))
        total   += len(labels)
    return correct / total

def adv_accuracy_scalar(m, epsilon=0.2, n_batches=5):
    total = correct = 0
    for i, (imgs, lbls) in enumerate(test_ds):
        if i >= n_batches: break
        adv = fgsm_attack(imgs, lbls, m, epsilon)
        preds = m(adv, training=False)
        correct += int(np.sum(np.argmax(preds.numpy(),1)==np.argmax(lbls.numpy(),1)))
        total   += lbls.shape[0]
    return correct / total

print("=== Final Comparison ===")
print("Standard model:")
clean_std = clean_accuracy(model, test_ds)
print(f"  Clean accuracy:       {clean_std*100:.2f}%")
evaluate_adversarial(model, test_ds, epsilon=0.2)

print("Robust model (adversarial training):")
clean_rob = clean_accuracy(robust_model, test_ds)
print(f"  Clean accuracy:       {clean_rob*100:.2f}%")
evaluate_adversarial(robust_model, test_ds, epsilon=0.2)

adv_std = adv_accuracy_scalar(model)
adv_rob = adv_accuracy_scalar(robust_model)

helper.plot_grouped_bars(
    group_labels=["Standard model", "Robust model"],
    bar_data=[[clean_std * 100, clean_rob * 100], [adv_std * 100, adv_rob * 100]],
    bar_labels=["Clean accuracy", "Adversarial accuracy"],
    ylabel="Accuracy (%)",
    title="Robustness–Accuracy Tradeoff (MNIST, ε=0.2)",
)